# Numerical work behind Peskin (1977): blood flow in the heart

**Goal:** understand the mathematics well enough to implement and validate a small immersed-boundary (IB) fluid solver yourself.

Primary reference: C. S. Peskin, *Numerical Analysis of Blood Flow in the Heart*, Journal of Computational Physics 25 (1977), 220–252, [DOI](https://doi.org/10.1016/0021-9991(77)90100-0).

> **Scope:** this notebook teaches the core numerical ideas with small runnable examples. It is not a line-by-line transcription or a reproduction of the paper's full heart simulation. Notation and a few algorithms below are simplified for clarity; where a modern demonstration differs from the paper's exact scheme, it is identified as such.

## How to work through it

For each section: (1) derive the equation on paper, (2) predict what the code should show, (3) run it, (4) change one parameter, and (5) explain the difference without looking at the code.

## 1. Physical equations and what each term means

For a Newtonian fluid of constant density \(\rho\) and dynamic viscosity \(\mu\), the incompressible Navier–Stokes equations with an added body-force density are

\[
\rho\left(\frac{\partial\mathbf u}{\partial t}
+(\mathbf u\cdot\nabla)\mathbf u\right)
=-\nabla p+\mu\nabla^2\mathbf u+\mathbf F,
\qquad \nabla\cdot\mathbf u=0.
\]

- \(\mathbf u(\mathbf x,t)\): velocity at a fixed point in space (Eulerian variable).
- \(p\): pressure; \(-\nabla p\) redistributes momentum to enforce incompressibility.
- \(\mu\nabla^2\mathbf u\): viscous momentum diffusion.
- \(\mathbf F\): force per unit volume; in IB it represents forces from immersed tissue.
- \(\nabla\cdot\mathbf u=0\): local volume conservation for an incompressible fluid.

**Why use one fluid grid?** The heart wall and valve leaflets move and deform, making a body-fitted grid difficult to maintain. Peskin represents the fluid on a fixed Cartesian grid and the tissue using moving material points. The two representations communicate through a smoothed delta function.

**Check:** in SI units, every term in the momentum equation has units of force per volume, \(\mathrm{N/m^3}\).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# A small set of illustrative blood-like parameters (not fitted to the paper).
rho = 1060.0       # kg/m^3
mu = 0.0035        # Pa s
nu = mu / rho      # m^2/s, kinematic viscosity
print(f"kinematic viscosity nu = {nu:.3e} m^2/s")
print("Momentum-equation units: kg/(m^2 s^2) = N/m^3")

## 2. Eulerian fluid grid versus Lagrangian boundary

The fluid velocity \(\mathbf u(\mathbf x,t)\) lives on a fixed spatial mesh. The tissue is described by material coordinates \(s\) and position \(\mathbf X(s,t)\). In the immersed-boundary method, a regularised delta kernel \(\delta_h\) transfers information between them:

\[
\mathbf F(\mathbf x,t)=\int \mathbf f(s,t)\,\delta_h(\mathbf x-\mathbf X(s,t))\,ds,
\]

\[
\frac{\partial\mathbf X}{\partial t}
=\int \mathbf u(\mathbf x,t)\,\delta_h(\mathbf x-\mathbf X(s,t))\,d\mathbf x.
\]

The first equation **spreads** boundary force onto nearby grid points. The second **interpolates** grid velocity to a boundary point. A discrete delta kernel replaces the singular Dirac delta with a compact, smooth stencil.

For a uniform one-dimensional grid, a common teaching kernel is the triangular hat function
\[
\phi(r)=\max(1-|r|,0),\qquad r=(x-X)/h.
\]
This is not claimed to be the exact improved kernel used in the 1977 paper; it is a simple kernel for understanding transfer and conservation.

In [ ]:
def hat_kernel(r):
    """Compact triangular kernel; support is |r| < 1."""
    return np.maximum(1.0 - np.abs(r), 0.0)

h = 1.0
grid = np.arange(-4, 5) * h
X = 0.35 * h
weights = hat_kernel((grid - X) / h)
print("weights:", np.round(weights, 4))
print("sum of weights:", weights.sum())
plt.figure(figsize=(7, 3))
plt.stem(grid, weights)
plt.xlabel("grid coordinate x")
plt.ylabel("kernel weight")
plt.title("Regularised delta stencil for one boundary point")
plt.grid(True, alpha=0.3)
plt.show()

### Derivation: discrete spreading and interpolation

With grid spacing \(h\), the one-dimensional approximation is

\[
F_i \approx \sum_k f_k\,\phi((x_i-X_k)/h)/h,
\qquad
U(X_k) \approx \sum_i u_i\,\phi((x_i-X_k)/h).
\]

The factor \(1/h\) in force spreading converts a point force into a force density in 1D. In 2D the corresponding factor is \(1/h^2\). A kernel whose weights sum to one preserves constants during interpolation. With consistent quadrature, spreading should preserve the total force.

**Important:** the precise scaling depends on whether the boundary force is defined per unit length, per boundary marker, or as a point force. Keep units explicit when turning this sketch into a solver.

In [ ]:
# Check constant-velocity interpolation and total force in 1D.
h = 0.2
xgrid = np.arange(-2.0, 2.0 + h/2, h)
X = 0.13
w = hat_kernel((xgrid - X) / h)
u_constant = np.full_like(xgrid, 2.75)
u_at_boundary = np.sum(u_constant * w)  # normalized hat weights on this uniform grid
F_point = 3.0
F_grid = F_point * w / h
print(f"kernel weight sum = {w.sum():.8f}")
print(f"interpolated constant velocity = {u_at_boundary:.8f}")
print(f"spread force integrated over grid = {np.sum(F_grid)*h:.8f}")
assert np.isclose(w.sum(), 1.0)
assert np.isclose(u_at_boundary, 2.75)
assert np.isclose(np.sum(F_grid)*h, F_point)

## 3. Elastic link forces: the simplest tissue model

Peskin discretises the boundary as material points connected by links. Let link \(AB\) have endpoints \(\mathbf X_A\) and \(\mathbf X_B\). Define

\[
\mathbf d=\mathbf X_B-\mathbf X_A,\qquad L=|\mathbf d|,\qquad
\mathbf q=\mathbf d/L.
\]

For a linear spring with resting length \(L_0\) and stiffness \(k_s\), a teaching force law is

\[
\mathbf F_A=k_s(L-L_0)\mathbf q,\qquad
\mathbf F_B=-\mathbf F_A.
\]

The sign convention means a stretched link pulls its endpoints together. These forces are equal and opposite, so their sum is zero: an internal link does not create net force on the entire structure.

The original paper uses a link tension law and discusses stability of the link network; a full valve or muscle model requires more than this single spring example.

In [ ]:
def spring_link_forces(xA, xB, rest_length, stiffness):
    """Return forces on endpoints A and B for a linear tensile/compressive spring."""
    d = np.asarray(xB, dtype=float) - np.asarray(xA, dtype=float)
    length = np.linalg.norm(d)
    if length == 0:
        raise ValueError("Link length is zero; direction is undefined.")
    force_A = stiffness * (length - rest_length) * d / length
    return force_A, -force_A

A = np.array([0.0, 0.0])
B = np.array([1.3, 0.4])
FA, FB = spring_link_forces(A, B, rest_length=1.0, stiffness=10.0)
print("force on A:", FA)
print("force on B:", FB)
print("net internal force:", FA + FB)
assert np.allclose(FA + FB, 0.0)

### Think before continuing

1. What happens to the force magnitude if the link is twice as stretched?
2. Why does the direction use \(\mathbf d/L\) rather than \(\mathbf d\)?
3. Why can a network of individually stable-looking springs still have an unstable deformation mode?
4. What changes if the link has a nonlinear tension–length law?

## 4. Why pressure is a Poisson problem

The velocity field must satisfy \(\nabla\cdot\mathbf u=0\). A typical projection step first forms a provisional velocity \(\mathbf u^*\), then corrects it using pressure:

\[
\mathbf u^{n+1}=\mathbf u^*-\frac{\Delta t}{\rho}\nabla p,
\]

and demands \(\nabla\cdot\mathbf u^{n+1}=0\). Taking divergence gives

\[
\nabla^2 p=\frac{\rho}{\Delta t}\nabla\cdot\mathbf u^*.
\]

This derivation is the core reason a pressure Poisson equation appears in incompressible solvers. Peskin's paper uses Chorin's finite-difference method and a fast Laplace solver; its detailed discrete operators and ordering are historical and not identical to every modern projection implementation.

On a periodic grid, pressure is only determined up to an additive constant. The right-hand side must have zero mean for a periodic Poisson equation to be solvable.

In [ ]:
# Use a compatible forward-gradient / backward-divergence pair.
# Their composition is exactly the 5-point periodic Laplacian below.
def grad_x_forward(a, h):
    return (np.roll(a, -1, axis=1) - a) / h

def grad_y_forward(a, h):
    return (np.roll(a, -1, axis=0) - a) / h

def div_x_backward(a, h):
    return (a - np.roll(a, 1, axis=1)) / h

def div_y_backward(a, h):
    return (a - np.roll(a, 1, axis=0)) / h

def divergence(u, v, h):
    return div_x_backward(u, h) + div_y_backward(v, h)

def laplacian_periodic(a, h):
    return ((np.roll(a, -1, axis=1) - 2*a + np.roll(a, 1, axis=1)) / h**2
          + (np.roll(a, -1, axis=0) - 2*a + np.roll(a, 1, axis=0)) / h**2)

N = 64
Lbox = 1.0
h = Lbox / N
x = np.arange(N) * h
Xg, Yg = np.meshgrid(x, x)
# A deliberately compressible provisional field, to demonstrate projection.
u_star = np.sin(2*np.pi*Xg) + 0.25*np.sin(2*np.pi*Yg)
v_star = 0.4*np.sin(2*np.pi*Yg)

## 5. Solve the periodic pressure equation with Fourier modes

For a periodic grid, the discrete Laplacian is diagonal in Fourier space. For each Fourier mode, the Poisson equation becomes multiplication by a scalar. The zero-frequency mode has eigenvalue zero, reflecting the arbitrary pressure constant; set that mode to zero after checking the source has zero mean.

This Fourier solve is a compact educational alternative to the specialised fast Laplace solver discussed by Peskin. It is not a reproduction of his exact solver implementation.

In [ ]:
def solve_poisson_periodic(rhs, h):
    """Solve discrete periodic Laplacian(p)=rhs using its Fourier eigenvalues."""
    rhs = np.asarray(rhs, dtype=float)
    rhs = rhs - rhs.mean()  # remove incompatible round-off / mean component
    ny, nx = rhs.shape
    kx = 2*np.pi*np.fft.fftfreq(nx, d=h)
    ky = 2*np.pi*np.fft.fftfreq(ny, d=h)
    KX, KY = np.meshgrid(kx, ky)
    # Eigenvalues of the 2nd-order, 5-point discrete Laplacian.
    lam_x = -4*np.sin(0.5*KX*h)**2 / h**2
    lam_y = -4*np.sin(0.5*KY*h)**2 / h**2
    lam = lam_x + lam_y
    rhs_hat = np.fft.fft2(rhs)
    p_hat = np.zeros_like(rhs_hat, dtype=complex)
    mask = np.abs(lam) > 1e-14
    p_hat[mask] = rhs_hat[mask] / lam[mask]
    p_hat[0, 0] = 0.0
    p = np.fft.ifft2(p_hat).real
    return p

rho_demo = 1.0
dt = 0.002
rhs = rho_demo / dt * divergence(u_star, v_star, h)
p = solve_poisson_periodic(rhs, h)
print("mean(rhs) after compatibility correction:", (rhs-rhs.mean()).mean())
print("Poisson residual (RMS):", np.sqrt(np.mean((laplacian_periodic(p,h) - (rhs-rhs.mean()))**2)))

## 6. Projection: reduce discrete divergence

The velocity correction is \(\mathbf u^{n+1}=\mathbf u^*-(\Delta t/\rho)\nabla_h p\). The operators in this demonstration are chosen deliberately so that the discrete divergence and gradient compose to the same 5-point Laplacian used in the pressure solve:

\[
D_h G_h p = L_h p.
\]

With \(L_h p=(\rho/\Delta t)D_h\mathbf u^*\), the corrected field should satisfy \(D_h\mathbf u^{n+1}\approx0\), up to round-off and solver tolerance. Here the gradient is forward-difference and divergence is backward-difference. This compatible pair is a pedagogical periodic-grid projection, not the exact staggered-grid implementation from the 1977 paper.

In [ ]:
u_new = u_star - dt/rho_demo * grad_x_forward(p, h)
v_new = v_star - dt/rho_demo * grad_y_forward(p, h)
div_before = divergence(u_star, v_star, h)
div_after = divergence(u_new, v_new, h)
print(f"RMS divergence before correction: {np.sqrt(np.mean(div_before**2)):.6e}")
print(f"RMS divergence after correction:  {np.sqrt(np.mean(div_after**2)):.6e}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
im0 = axes[0].imshow(div_before, origin="lower", extent=[0,Lbox,0,Lbox])
axes[0].set_title("Before pressure correction")
axes[1].imshow(div_after, origin="lower", extent=[0,Lbox,0,Lbox])
axes[1].set_title("After pressure correction")
for ax in axes:
    ax.set_xlabel("x")
    ax.set_ylabel("y")
plt.colorbar(im0, ax=axes, shrink=0.8, label="divergence")
plt.tight_layout()
plt.show()
assert np.sqrt(np.mean(div_after**2)) < 1e-8 * np.sqrt(np.mean(div_before**2))

### Debugging lesson: operator compatibility

If the divergence does not fall enough, do not immediately assume the pressure equation is wrong. Check whether the discrete operators satisfy the same algebraic relationship as the continuous operators. A common stable arrangement uses a staggered grid and compatible discrete divergence/gradient operators. The original paper's finite-difference scheme is carefully structured; reproducing that structure matters.

**Exercise:** derive the discrete pressure equation directly from the discrete velocity correction you actually implement. Do not mix a pressure Laplacian from one stencil with a gradient from another without checking the resulting operator product.

## 7. A miniature immersed-boundary coupling experiment

This small example shows the *data-transfer logic*, not a complete fluid solver: one boundary point spreads a force over nearby grid cells using a compact kernel. A real IB solver would then include that force density in the momentum update, solve the pressure constraint, interpolate the updated velocity back to all boundary points, and update the boundary configuration. These steps are coupled over time.

In [ ]:
# 2D tensor-product hat kernel for a single immersed point.
def hat2(r):
    return np.maximum(1.0 - np.abs(r[..., 0]), 0.0) * np.maximum(1.0 - np.abs(r[..., 1]), 0.0)

N = 48
Lbox = 1.0
h = Lbox / N
coords = np.arange(N) * h
Xg, Yg = np.meshgrid(coords, coords)
Xb = np.array([0.48, 0.53])
Fb = np.array([1.0, -0.5])  # illustrative point force
rx = (Xg - Xb[0]) / h
ry = (Yg - Xb[1]) / h
weights2 = np.maximum(1-np.abs(rx),0) * np.maximum(1-np.abs(ry),0)
# Divide by h^2 to turn a point force into force density in 2D.
Fx = Fb[0] * weights2 / h**2
Fy = Fb[1] * weights2 / h**2
integrated_force = np.array([Fx.sum()*h**2, Fy.sum()*h**2])
print("original point force:", Fb)
print("integral of spread force density:", integrated_force)
print("relative transfer error:", np.linalg.norm(integrated_force-Fb)/np.linalg.norm(Fb))
fig, ax = plt.subplots(figsize=(5,4))
im = ax.imshow(np.hypot(Fx,Fy), origin="lower", extent=[0,Lbox,0,Lbox])
ax.plot(Xb[0], Xb[1], "ro", label="immersed point")
ax.set_title("Force density spread onto the fixed grid")
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.legend()
plt.colorbar(im, ax=ax, label="force-density magnitude")
plt.show()

## 8. From the components to a full time step

The logic of a coupled immersed-boundary simulation is:

1. **Boundary mechanics:** use the current marker positions to compute elastic / prescribed tissue forces.
2. **Spread:** transfer those forces to the fixed fluid grid with the regularised delta kernel.
3. **Fluid momentum:** advance the Navier–Stokes momentum terms to obtain a provisional velocity.
4. **Pressure constraint:** solve the pressure Poisson/Laplace problem and correct velocity to satisfy incompressibility.
5. **Interpolate:** transfer the fluid velocity back to boundary markers.
6. **Move boundary:** update marker positions; because the boundary force depends on the new configuration, some schemes require an implicit nonlinear solve.
7. **Repeat:** track stability, force balance, divergence, and time-step sensitivity.

In the 1977 paper, the boundary configuration is obtained from a nonlinear fixed-point system, solved using Newton's method. The fixed topology of the boundary link structure lets the Jacobian have exploitable sparsity; Peskin discusses Cholesky factorisation under a stability assumption on the link structure. The fluid step uses finite differences and a fast Laplace solver. The paper also adds a source/sink model to represent pulmonary venous return and accommodate changing enclosed volume in a periodic box.

Those are not incidental implementation details: they are the difference between a set of equations and a functioning simulator.

## 9. Validation exercises (do these before adding heart geometry)

### A. Conservation and transfer
- Move the immersed point continuously through a grid cell. Does the total spread force stay approximately constant?
- Test constant grid velocities. Does interpolation return the same constant at every boundary location?
- Try a narrower or wider kernel. Which properties break first?

### B. Elasticity
- Stretch one link and check that the endpoint forces are equal and opposite.
- Make a closed triangle of links. Perturb one vertex and inspect whether the force points restore or destabilise the shape.
- Vary stiffness. Explain why larger stiffness can force a smaller stable time step or require an implicit solve.

### C. Fluid solver
- Start from a divergence-free periodic velocity field and verify that the pressure correction does not introduce large divergence.
- Check the Poisson residual and the mean of its right-hand side.
- Compare results at two grid resolutions and two time steps. Do not call a simulation validated merely because it produces a plausible-looking plot.

### D. Coupling
- Start with a stationary boundary and a simple fluid field.
- Add a single elastic link before adding a closed valve.
- Log maximum speed, RMS divergence, total force, and boundary energy at every step.
- Add muscle activation and source/sink flow only after the simpler problem passes tests.

## 10. What this notebook does not yet solve

- It does not calculate a realistic full cardiac cycle.
- It does not reproduce the paper's full natural or prosthetic valve geometries or its numerical results.
- It does not implement the complete nonlinear boundary solve or the source/sink pressure-resistance model.
- The hat kernel and Fourier Poisson solver are teaching substitutes, not exact reproductions of the paper's improved delta kernel and fast Laplace implementation.
- The projection section is pedagogical and uses simplified collocated-grid operators; a production solver needs a compatible discretisation and stability analysis.

These limitations are intentional. The next engineering step is to implement each operator as a separately testable module and compare it against analytical or manufactured solutions before building the full coupled heart model.

## References

1. C. S. Peskin (1977), “Numerical Analysis of Blood Flow in the Heart,” *Journal of Computational Physics*, 25, 220–252. [DOI](https://doi.org/10.1016/0021-9991(77)90100-0); [publisher page](https://www.sciencedirect.com/science/article/pii/0021999177901000).
2. C. S. Peskin (2002), “The Immersed Boundary Method,” *Acta Numerica*. [Cambridge Core](https://www.cambridge.org/core/journals/acta-numerica/article/immersed-boundary-method/95ECDAC5D1824285563270D6DD70DA9A).